<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Math & Python for AI** · Week 4 · Calculus and Your First Model · Lesson 23 of 24

# Linear Regression from Scratch

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/math-python-for-ai).

**Learning objective:** Build a linear regression model with your own gradient descent, measure its error, compare it with NumPy's answer, and evaluate it on test data.

## The model

We predict a moto fare from the trip distance with a straight line:

> predicted fare = w × distance + b

- **w** (weight, the slope): extra RWF per kilometre.
- **b** (bias, the intercept): the starting fare.

Training means finding the w and b that make predictions as close as possible to the real fares.

Run the setup cell:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 300
distance = rng.uniform(1, 15, n).round(1)
rain = rng.random(n) < 0.25
hour = rng.integers(6, 23, n)
fare = (400 + 230 * distance + np.where(rain, 300, 0) + rng.normal(0, 150, n)).round(-1)
minutes = (distance * 2.8 + rng.normal(0, 3, n) + np.where(rain, 4, 0)).clip(3).round()
trips = pd.DataFrame({"distance_km": distance, "minutes": minutes, "fare_rwf": fare,
                      "hour": hour, "raining": rain})

## Split into training and test data

In [ ]:
idx = rng.permutation(n)
train, test = trips.iloc[idx[:240]], trips.iloc[idx[240:]]
x_train, y_train = train["distance_km"].to_numpy(), train["fare_rwf"].to_numpy()
x_test, y_test = test["distance_km"].to_numpy(), test["fare_rwf"].to_numpy()
print(len(train), len(test))

We learn from 240 trips and keep 60 aside to test honestly (AI Foundations, Lesson 13).

## The loss: mean squared error

> MSE = average of (prediction − real value)²

In [ ]:
def predict(x, w, b):
    return w * x + b

def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

print(round(mse(y_train, predict(x_train, 0, 0))))   # a terrible starting guess

## The gradients

Using the derivative rules from Lesson 21:

> ∂MSE/∂w = 2 × average((prediction − y) × x)
> ∂MSE/∂b = 2 × average(prediction − y)

## Gradient descent

In [ ]:
w, b = 0.0, 0.0
lr = 0.01
history = []
for epoch in range(5000):
    error = predict(x_train, w, b) - y_train
    dw = 2 * np.mean(error * x_train)
    db = 2 * np.mean(error)
    w -= lr * dw
    b -= lr * db
    history.append(mse(y_train, predict(x_train, w, b)))

print(f"w = {w:.1f} RWF per km, b = {b:.1f} RWF")

w should come out at about **225 RWF per km** and b at about **520 RWF** — close to the rule used to simulate the data (230 RWF per km and a 400 RWF start). b is a little higher because about a quarter of the trips include a 300 RWF rain charge that this model cannot see. The model discovered the pricing rule from the examples alone.

In [ ]:
plt.plot(history)
plt.yscale("log")
plt.title("Training loss (MSE)")
plt.xlabel("epoch")
plt.show()

## Check against NumPy

NumPy can solve a straight-line fit directly:

In [ ]:
w_np, b_np = np.polyfit(x_train, y_train, 1)
print(round(w_np, 1), round(b_np, 1))

Your gradient descent should agree to within about 1 RWF. For a straight line there is an exact formula; gradient descent matters because it also works for models with no exact formula — like neural networks.

## Evaluate on the test set

In [ ]:
pred_test = predict(x_test, w, b)
mae = np.mean(np.abs(pred_test - y_test))
r2 = 1 - np.sum((y_test - pred_test) ** 2) / np.sum((y_test - y_test.mean()) ** 2)
print(f"Test MAE: {mae:.0f} RWF   Test R²: {r2:.3f}")

plt.scatter(x_test, y_test, alpha=0.6, label="real fares (test)")
xs = np.linspace(1, 15, 50)
plt.plot(xs, predict(xs, w, b), color="red", label="model")
plt.xlabel("Distance (km)")
plt.ylabel("Fare (RWF)")
plt.legend()
plt.show()

- **MAE** — on average, the prediction is off by this many RWF.
- **R²** — the share of variation in fares the model explains (1 = perfect).

## What the model cannot see

Part of the remaining error comes from **rain**: rainy trips cost about 300 RWF more, and our model only knows distance. Adding more features (multiple linear regression) is your final project.

## Key terms

- **Linear regression** — predicting a number with a straight-line model w × x + b.
- **Weight / bias** — slope / intercept of the line.
- **MSE** — mean squared error, the loss we minimise.
- **Epoch** — one pass of gradient descent over the training data.
- **MAE** — mean absolute error, easy to explain in real units.
- **R²** — share of variation explained by the model.

**Takeaway:** You built a real machine-learning model from scratch: a prediction function, a loss, gradients and gradient descent. It learned the fare rule from data and was checked honestly on test data.

## Try it yourself

1. Change the learning rate to 0.03, then 0.001. What happens to training?
2. Predict the fare for a 7.5 km trip with your model.
3. Train a second model that predicts `minutes` from `distance_km`. How many minutes per kilometre does it learn?
4. Compute the test MAE separately for rainy and dry trips. Which is worse, and why?

---
Finished? Go back to [Math & Python for AI on WAMUGII](https://wamugii.kento.rw/courses/math-python-for-ai), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*